In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install -q peft datasets trl transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 697.4/697.4 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 38.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 14.8 MB/s eta 0:00:00


In [ ]:
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments
)
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig
import torch
from datasets import load_dataset

In [ ]:
train_path = '/content/drive/MyDrive/hybrid_ner_extractor/data/processed/train.jsonl'
val_path = '/content/drive/MyDrive/hybrid_ner_extractor/data/processed/val.jsonl'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True
)

In [ ]:
model_name = 'Qwen/Qwen2.5-0.5B'
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
    dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM'
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 2,162,688 || all params: 496,195,456 || trainable%: 0.4359


In [ ]:
dataset = load_dataset('json', data_files={'train': train_path, 'validation': val_path})

sft_config = SFTConfig(
    output_dir='./llm_finetuned',
    max_length=512,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_steps=50,
    max_steps=800,
    fp16=False,
    bf16=False,
    logging_steps=10,
    save_strategy='steps',
    save_steps=200,
    eval_strategy='steps',
    eval_steps=50,
    report_to='none',
    dataset_text_field='text'
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    args=sft_config,
    train_dataset=dataset['train'],
    eval_dataset=dataset['validation']
)

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/600 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/600 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/50 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/50 [00:00<?, ? examples/s]

In [ ]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss,Validation Loss
50,0.510117,0.396999
100,0.231062,0.247708
150,0.221246,0.212311
200,0.194614,0.193166
250,0.174691,0.186975
300,0.177211,0.177664
350,0.162598,0.174111
400,0.157882,0.172457
450,0.144822,0.165925
500,0.150355,0.168529


TrainOutput(global_step=800, training_loss=0.24155993655323982, metrics={'train_runtime': 1663.7677, 'train_samples_per_second': 1.923, 'train_steps_per_second': 0.481, 'total_flos': 832649961580800.0, 'train_loss': 0.24155993655323982})

In [ ]:
model.save_pretrained('/content/drive/MyDrive/hybrid_ner_extractor/models/llm_lora')
tokenizer.save_pretrained('/content/drive/MyDrive/hybrid_ner_extractor/models/llm_lora')

('/content/drive/MyDrive/hybrid_ner_extractor/models/llm_lora/tokenizer_config.json',
 '/content/drive/MyDrive/hybrid_ner_extractor/models/llm_lora/chat_template.jinja',
 '/content/drive/MyDrive/hybrid_ner_extractor/models/llm_lora/tokenizer.json')